<a href="https://colab.research.google.com/github/agbizbuz/learning-ai-ds-ml/blob/main/simplilearn_ml_python/stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ucimlrepo

### Improvements Implemented:
1. **Fixed XGBoost Crash**: The original Wine dataset labels were `1, 2, 3`, but modern XGBoost expects `0, 1, 2`. We now adjust labels (`y - 1`).
2. **Replaced `vecstack`**: Removed the external `vecstack` dependency in favor of the native `sklearn.ensemble.StackingClassifier`. This is the industry standard for stacked ensembles.
3. **Added Feature Scaling**: Added `StandardScaler` pipeline. KNN (distance-based) is heavily impacted by feature magnitude and requires scaling.
4. **Baseline Comparisons**: Added a cell to compare individual model performance against the final stacked ensemble accuracy to verify the value of ensembling.

In [5]:
import numpy as np
from ucimlrepo import fetch_ucirepo 

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [6]:
# fetch dataset
wine = fetch_ucirepo(id=109) 

# Retrieve features and target
X = wine.data.features.values

# Fix 1: Adjust target labels from 1,2,3 to 0,1,2 (Required for XGBoost 2.0+)
y = wine.data.targets.values.ravel() - 1 

print(f"Data shape: {X.shape}")
print(f"Target classes: {np.unique(y)}")

Data shape: (178, 13)
Target classes: [0 1 2]


In [7]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# Scale features - Critical for KNN and ensures LogisticRegression is optimized effectively
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
# Define the base estimators (Weak Learners)
estimators = [
    ('knn', KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=0, n_jobs=-1)),
    ('xgb', XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=0, n_jobs=-1))
]

In [10]:
# Define the Stacking Ensemble
# StackingClassifier automatically handles out-of-fold (OOF) predictions via the 'cv' parameter
meta_learner = LogisticRegression(max_iter=1000)

stacker = StackingClassifier(
    estimators=estimators, 
    final_estimator=meta_learner, 
    cv=5, # Number of folds for out-of-fold predictions (generates features for meta_learner)
    stack_method='predict_proba' # Passes probability estimates (0.0 to 1.0) to the meta-learner
)

# Fit the Stacking ensemble using scaled features
stacker.fit(X_train_scaled, y_train)

y_pred_stack = stacker.predict(X_test_scaled)

In [11]:
# Evaluate the Stacking Ensemble
print(f"--- Stacking Ensemble Accuracy: {accuracy_score(y_test, y_pred_stack):.4f} \n")
print(classification_report(y_test, y_pred_stack))

# Compare baseline models to see if the ensemble actually helped
print("\n--- Base Models Comparison (with Scaling) ---")
for name, model in estimators:
    model.fit(X_train_scaled, y_train)
    y_pred_base = model.predict(X_test_scaled)
    print(f"{name:10s} Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")

--- Stacking Ensemble Accuracy: 1.0000 

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36


--- Base Models Comparison (with Scaling) ---
knn        Accuracy: 0.9444
rf         Accuracy: 1.0000
xgb        Accuracy: 0.9722
